# Instagram & Facebook Sentiment Analysis

This notebook demonstrates a complete sentiment analysis pipeline for social media comments.

## 1. Import Required Libraries

Import all the necessary libraries for data analysis, machine learning, and visualization.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
import re

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

print("Libraries imported successfully!")

## 2. Load Dataset from CSV

Load the comments dataset from the CSV file.

In [ ]:
# Load the dataset
df = pd.read_csv('../data/comments.csv')

# Display basic information
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

## 3. Clean Text Data

Clean the text data by removing special characters, URLs, and converting to lowercase.

In [ ]:
def clean_text(text):
    """
    Clean the input text by removing special characters, URLs, and extra spaces.
    """
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Convert to lowercase
    text = text.lower()
    return text

# Apply text cleaning
df['cleaned_comment'] = df['comment'].apply(clean_text)

print("Text cleaning completed!")
print("\nSample cleaned comments:")
for i in range(3):
    print(f"Original: {df['comment'][i]}")
    print(f"Cleaned: {df['cleaned_comment'][i]}")
    print("-" * 50)

## 4. Remove Stopwords

Remove common stopwords from the cleaned text using NLTK.

In [ ]:
def remove_stopwords(text):
    """
    Remove stopwords from the text.
    """
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(text)
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

# Apply stopword removal
df['no_stopwords'] = df['cleaned_comment'].apply(remove_stopwords)

print("Stopword removal completed!")
print("\nSample after stopword removal:")
for i in range(3):
    print(f"Cleaned: {df['cleaned_comment'][i]}")
    print(f"No stopwords: {df['no_stopwords'][i]}")
    print("-" * 50)

## 5. Tokenization

Tokenize the text and apply lemmatization.

In [ ]:
def tokenize_and_lemmatize(text):
    """
    Tokenize the text and apply lemmatization.
    """
    lemmatizer = WordNetLemmatizer()
    words = word_tokenize(text)
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(lemmatized_words)

# Apply tokenization and lemmatization
df['processed_comment'] = df['no_stopwords'].apply(tokenize_and_lemmatize)

print("Tokenization and lemmatization completed!")
print("\nSample processed comments:")
for i in range(3):
    print(f"Original: {df['comment'][i]}")
    print(f"Processed: {df['processed_comment'][i]}")
    print("-" * 50)

## 6. TF-IDF Vectorization

Convert the processed text into numerical features using TF-IDF vectorization.

In [ ]:
# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['processed_comment'])
y = df['sentiment']

print("TF-IDF vectorization completed!")
print(f"Feature matrix shape: {X.shape}")
print(f"Number of features: {len(vectorizer.get_feature_names_out())}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

## 7. Train Machine Learning Model with Logistic Regression

Train a Logistic Regression model on the vectorized data.

In [ ]:
# Train Logistic Regression model
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

# Make predictions
lr_predictions = lr_model.predict(X_test)

print("Predictions completed!")

## 8. Train Machine Learning Model with Naive Bayes

Train a Naive Bayes model on the vectorized data.

In [ ]:
# Train Naive Bayes model
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

print("Naive Bayes model trained successfully!")

# Make predictions
nb_predictions = nb_model.predict(X_test)

print("Predictions completed!")

## 9. Show Accuracy Score

Calculate and display the accuracy scores for both models.

In [ ]:
# Calculate accuracy scores
lr_accuracy = accuracy_score(y_test, lr_predictions)
nb_accuracy = accuracy_score(y_test, nb_predictions)

print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")
print(f"Naive Bayes Accuracy: {nb_accuracy:.4f}")

# Detailed classification report
print("\nLogistic Regression Classification Report:")
print(classification_report(y_test, lr_predictions))

print("\nNaive Bayes Classification Report:")
print(classification_report(y_test, nb_predictions))

## 10. Display Confusion Matrix

Generate and display confusion matrices for both models.

In [ ]:
# Create confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Logistic Regression Confusion Matrix
lr_cm = confusion_matrix(y_test, lr_predictions)
sns.heatmap(lr_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['negative', 'neutral', 'positive'],
            yticklabels=['negative', 'neutral', 'positive'], ax=axes[0])
axes[0].set_title('Logistic Regression Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Naive Bayes Confusion Matrix
nb_cm = confusion_matrix(y_test, nb_predictions)
sns.heatmap(nb_cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['negative', 'neutral', 'positive'],
            yticklabels=['negative', 'neutral', 'positive'], ax=axes[1])
axes[1].set_title('Naive Bayes Confusion Matrix')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

## 11. Create Charts for Sentiment Distribution

Create visualizations showing the distribution of sentiments in the dataset.

In [ ]:
# Sentiment distribution
sentiment_counts = df['sentiment'].value_counts()

# Bar chart
plt.figure(figsize=(10, 6))
sentiment_counts.plot(kind='bar', color=['#ff9999','#66b3ff','#99ff99'])
plt.title('Sentiment Distribution in Dataset')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

# Pie chart
plt.figure(figsize=(8, 8))
plt.pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%',
        colors=['#ff9999','#66b3ff','#99ff99'])
plt.title('Sentiment Distribution (Pie Chart)')
plt.axis('equal')
plt.show()

## 12. Data Visualizations: Word Cloud

Generate a word cloud from all the comments in the dataset.

In [ ]:
# Generate word cloud
all_text = ' '.join(df['processed_comment'])

wordcloud = WordCloud(width=800, height=400, background_color='white',
                      max_words=100, contour_width=3, contour_color='steelblue')

wordcloud.generate(all_text)

# Display word cloud
plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Processed Comments', fontsize=16)
plt.show()